# Phase 1 — undefended English baseline, on a free Colab T4

Runs top to bottom and ends Phase 1's compute: closes the Phase 0 live-call exit
criterion, then produces `uASR(en, undefended)` and `Utility(en, undefended)` over the
150-case stratified subset.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. That is the whole procedure: every step skips work whose results already exist (for this model, attack and corpus) and saves to Drive when it finishes, so re-running costs minutes, not hours, and an exhausted quota loses at most the step in progress. Without a GPU, cell 6 fails.

What this does, in order:

1. checks the GPU and prints its compute capability
2. mounts Drive (optional, but it is how the cache survives a disconnect)
3. gets `agentfw` onto the machine — from a Drive zip, an upload, or a git URL
4. builds a Python 3.10 environment and installs vLLM, AgentDojo at its pinned SHA, and `agentfw` into it
5. serves `Qwen2.5-14B-Instruct-AWQ` (falls back to the 7B) with an OpenAI-compatible endpoint
6. **verifies tool calling** — everything downstream is meaningless if this fails
7. `phase0_smoke --backend local` → closes Phase 0
8. `phase1_baseline --limit 10` → sanity gate before the real spend of time
9. `phase1_baseline` over all 150 cases, attack pass + clean pass
10. prints the numbers and writes a dated `RESULTS.md` entry you can paste
11. Phase 2 head start: the capability probe in all five languages (four as pilots)
12. saves cache, traces and every result file to Drive

**Honest caveats, read them once:**

- A free T4 has 16 GB and fp16 weights for a 7B model are ~15.2 GB, so the fp16 recipe
  in `docs/KAGGLE.md` (written for Kaggle's 2×T4) **cannot** work here. This notebook
  serves the **AWQ 4-bit** build instead. That is a different model artifact from the
  one the docs name — record it in `RESULTS.md`, because quantization can move a
  capability ceiling and `C(L)` is the project's headline.
- Colab free disconnects on idle and caps sessions. The response cache makes a resumed
  run free, so a disconnect costs wall-clock, not results — provided you let cell 3
  restore the cache from Drive.
- The vLLM version is pinned because T4 is Turing (sm75) and newer vLLM releases have
  dropped or destabilised Turing support. If the pin fails to install, cell 5 tells you
  what to try.

## 1 · GPU check

In [ ]:
!nvidia-smi
import subprocess, sys
try:
    cap = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
        text=True).strip()
    print("\nGPU:", cap)
    if "7.5" in cap:
        print("Turing (sm75). The pinned vLLM version below is chosen for this.")
except Exception as e:
    sys.exit(f"No GPU visible ({e}). Runtime -> Change runtime type -> T4 GPU.")

## 2 · Config

The only cell you should need to edit.

In [ ]:
# Model. 14B AWQ 4-bit (~9.3 GiB of weights). The 7B baseline (RESULTS.md, 2026-10-02)
# completed only 30% of tasks and 44% of attacker goals, leaving 5 successful attacks:
# too few to compare defenses. 14B is the same Qwen2.5 family, so the same vLLM works.
MODEL     = "Qwen/Qwen2.5-14B-Instruct-AWQ"
QUANT     = "awq"
REVISION  = "main"          # pin a commit SHA here once you have one, then never move it

# Memory on one 16 GB T4 is tight for 14B: weights ~9.3 GiB leave ~3-4 GiB of KV cache.
# 12288 tokens still covers the largest prompts (workspace/travel tool schemas ~4-5k);
# --enforce-eager skips CUDA-graph capture, which would otherwise take ~1 GiB.
MAX_LEN    = 12288
GPU_UTIL   = 0.95
EXTRA_ARGS = ["--enforce-eager"]

# Fallback if 14B will not load: the 7B AWQ build, which is known to work on this setup.
# If the fallback is used, say so in RESULTS.md -- it changes the capability ceiling.
FALLBACK_MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
FALLBACK_QUANT = "awq"

# vLLM pinned for Turing. If this install fails, try "vllm==0.7.3", then plain "vllm".
VLLM_SPEC = "vllm==0.6.6.post1"

# Attack. AgentDojo's standard template; "direct" = bare goal, as the first baselines ran.
ATTACK = "important_instructions"

# Every step below skips work whose results already exist for this model, attack and
# corpus. Set True only to deliberately recompute (the response cache still applies).
RERUN = False

# Run shape
WORKERS = 8          # vLLM batches continuously, so concurrency is what buys throughput
SMOKE_N = 10         # cases in the pre-flight gate
FULL_N  = None       # None = all 150

# Paths
REPO_DIR      = "/content/agentfw"
DRIVE_DIR     = "/content/drive/MyDrive/agentfw"   # cache + results persist here
USE_DRIVE     = True
GIT_URL       = ""   # e.g. "https://github.com/you/agentfw.git"; blank = zip or upload
AGENTDOJO_SHA = "089ed468cf3ed0322acc66b0211f26d9d90dbf60"

# The response cache keys on the model id, so changing MODEL starts the cache cold.
print("model:", MODEL, "| workers:", WORKERS, "| cases:", FULL_N or "all")

## 3 · Drive (optional but recommended)

Holds `cache.tgz` between sessions. A resumed run then costs no GPU time for anything it
already computed.

In [ ]:
import os
DRIVE_OK = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        DRIVE_OK = True
        print("drive ready:", DRIVE_DIR)
        print("contents:", os.listdir(DRIVE_DIR) or "(empty)")
    except Exception as e:
        print("drive unavailable, continuing without persistence:", e)
else:
    print("drive disabled by config")

## 4 · Get the repo

Three routes, tried in order:

1. `GIT_URL` if you set one in cell 2
2. `agentfw.zip` in `DRIVE_DIR`
3. an interactive upload

For route 2, zip the project folder on your machine and drop it in Drive as
`MyDrive/agentfw/agentfw.zip`. The zip may contain the project at its root or inside a
single top-level folder — both are handled. `vendor/agentdojo` is not needed in the zip;
cell 5 clones it at the pinned SHA.

In [ ]:
import os, shutil, subprocess, zipfile, glob

def _flatten(dest):
    """If the zip had a single wrapper folder, lift its contents up."""
    entries = [e for e in os.listdir(dest) if e != "__MACOSX"]
    if len(entries) == 1 and os.path.isdir(os.path.join(dest, entries[0])):
        inner = os.path.join(dest, entries[0])
        if os.path.exists(os.path.join(inner, "pyproject.toml")):
            for e in os.listdir(inner):
                shutil.move(os.path.join(inner, e), os.path.join(dest, e))
            os.rmdir(inner)

if os.path.exists(os.path.join(REPO_DIR, "pyproject.toml")):
    print("repo already present at", REPO_DIR)
elif GIT_URL:
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, REPO_DIR], check=True)
else:
    zips = sorted(glob.glob(os.path.join(DRIVE_DIR, "*.zip"))) if DRIVE_OK else []
    if zips:
        src = zips[0]
        print("unzipping", src)
    else:
        from google.colab import files
        print("Upload a zip of the agentfw project folder:")
        up = files.upload()
        src = "/content/" + next(iter(up))
    os.makedirs(REPO_DIR, exist_ok=True)
    with zipfile.ZipFile(src) as z:
        z.extractall(REPO_DIR)
    _flatten(REPO_DIR)

assert os.path.exists(os.path.join(REPO_DIR, "harness", "runner.py")), \
    f"{REPO_DIR} does not look like agentfw: {os.listdir(REPO_DIR)}"

# Restore the cache and traces from a previous session, if any.
tgz = os.path.join(DRIVE_DIR, "cache.tgz") if DRIVE_OK else ""
if tgz and os.path.exists(tgz):
    subprocess.run(["tar", "xzf", tgz, "-C", REPO_DIR], check=True)
    print("cache restored from", tgz)
else:
    print("no cache.tgz in Drive — starting cold (expected on the first session)")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
print(sorted(os.listdir()))

def save_to_drive(note=""):
    """Push cache, traces and results to Drive. Called after every long step, so a
    disconnect or an exhausted quota costs at most the step that was running."""
    import glob, re
    subprocess.run(["tar", "czf", "/content/cache.tgz", ".cache", "traces", "analysis/out"], check=False)
    size = os.path.getsize("/content/cache.tgz") / 1e6
    if not DRIVE_OK:
        print(f"cache.tgz {size:.1f} MB (Drive not mounted -- run the last cell to download it)")
        return
    # copy then rename: a session killed mid-copy must not leave a half-written archive
    subprocess.run(["cp", "/content/cache.tgz", f"{DRIVE_DIR}/cache.tgz.tmp"], check=True)
    os.replace(f"{DRIVE_DIR}/cache.tgz.tmp", f"{DRIVE_DIR}/cache.tgz")
    served = globals().get("SERVED", MODEL)
    dest = f"{DRIVE_DIR}/results/{re.sub(r'[^A-Za-z0-9._-]+', '-', served)}"
    os.makedirs(dest, exist_ok=True)
    files = glob.glob("analysis/out/phase1_*") + glob.glob("analysis/out/phase2_probe_*.json")
    for f in files:
        subprocess.run(["cp", f, f"{dest}/"], check=False)
    print(f"saved to Drive{' (' + note + ')' if note else ''}: cache.tgz {size:.1f} MB, {len(files)} result files -> {dest}")


## 5 · Install

Everything goes into a **separate Python 3.10 environment** at `/content/venv`, because
Colab's own Python (3.13 as of Oct 2026) is too new for the vLLM version pinned for the T4.
Every later cell runs its programs with `virtual_env_python`, so **after any runtime
restart, re-run this cell first** — it is quick once the environment exists.

The first session that gets as far as a working tool call (cell 7) saves the exact package versions to
`MyDrive/agentfw/requirements-colab.lock`; later sessions install exactly those, so a new
release of some package cannot break a later phase.

First time: 5–10 minutes. It may suggest restarting the runtime — **do not**, just continue.


In [ ]:
import subprocess, sys, os

# Colab's own Python moves (3.13 as of Oct 2026) and vLLM pinned for the T4 needs
# <=3.12, so everything runs in a separate Python 3.10 environment. Every later
# cell launches programs with `virtual_env_python`, never `sys.executable`.
VENV = "/content/venv"
virtual_env_python = f"{VENV}/bin/python"
LOCK = f"{DRIVE_DIR}/requirements-colab.lock" if DRIVE_OK else None

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        print("".join((r.stdout + r.stderr).splitlines(keepends=True)[-40:]))
        raise RuntimeError(f"failed: {' '.join(cmd)} -- see the output above")
    return r.stdout

def stream(cmd):
    """Run a program and show its output in this cell as it runs.

    Colab does not display output from a plain subprocess.run(), so a failing
    script would show nothing but its exit code.
    """
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    return subprocess.CompletedProcess(cmd, p.wait())

def pip(*args):
    run([virtual_env_python, "-m", "pip", "install", "-q", *args])

# 1. Python 3.10. uv downloads a standalone interpreter, so this keeps working
#    even on a Colab image whose apt no longer carries python3.10.
if not os.path.exists(virtual_env_python):
    print("creating the Python 3.10 environment ...")
    run(["python3", "-m", "pip", "install", "-q", "uv"])
    run(["python3", "-m", "uv", "venv", "--seed", "--python", "3.10", VENV])
print("Colab Python:", sys.version.split()[0])
print("venv Python: ", run([virtual_env_python, "--version"]).strip())

# 2. AgentDojo at its pinned commit (vendor/ is gitignored, so a fresh copy has none).
if not os.path.exists("vendor/agentdojo/pyproject.toml"):
    os.makedirs("vendor", exist_ok=True)
    run(["git", "clone", "--quiet", "--depth", "50",
         "https://github.com/ethz-spylab/agentdojo.git", "vendor/agentdojo"])
    run(["git", "-C", "vendor/agentdojo", "checkout", "--quiet", AGENTDOJO_SHA])
sha = run(["git", "-C", "vendor/agentdojo", "rev-parse", "HEAD"]).strip()
assert sha == AGENTDOJO_SHA, f"AgentDojo at {sha}, expected {AGENTDOJO_SHA}"
print("agentdojo pinned at", sha)

# 3. Packages. The first good session writes a lock file to Drive; every later
#    session installs exactly those versions, so an upstream release can never
#    break a later phase. Delete the lock in Drive only to deliberately upgrade.
if LOCK and os.path.exists(LOCK):
    locked = open(LOCK).read()
    t = [l for l in locked.splitlines() if l.lower().startswith("transformers==")]
    if not t or not t[0].split("==")[1].startswith("4."):
        print("discarding", LOCK, "-- it records", t or "no transformers", "which breaks vLLM 0.6.6")
        os.remove(LOCK)
if LOCK and os.path.exists(LOCK):
    print("installing the exact versions from", LOCK, "...")
    pip("-r", LOCK)
else:
    # vLLM first: it pins torch, and anything installed before it could be stomped.
    print(f"installing {VLLM_SPEC} and dependencies (first time: 5-10 min) ...")
    # vLLM 0.6.x declares transformers>=4.45.2 with no upper bound, and
    # transformers 5.x removed `all_special_tokens_extended`, which this vLLM
    # still reads -- the server then dies loading the tokenizer. Pin the
    # transformers of vLLM 0.6.6's own era, in the same resolve as vLLM.
    pip(VLLM_SPEC, "transformers>=4.45.2,<4.48")
    # AgentDojo's own dependency list drags in langchain for a pipeline we do not
    # use, so it goes in with --no-deps and its real runtime imports are listed
    # here. agentdojo.agent_pipeline imports every provider SDK at load time,
    # hence anthropic/cohere/google-genai: client libraries only, no paid calls.
    VLLM_PINS = "/content/vllm-constraints.txt"
    with open(VLLM_PINS, "w") as f:
        f.write(run([virtual_env_python, "-m", "pip", "freeze", "--exclude-editable"]))
    pip("-c", VLLM_PINS, "pydantic[email]>=2.7.1", "docstring-parser>=0.15", "tenacity>=8.2.3",
        "pyyaml>=6.0.1", "click>=8.1.7", "deepdiff>=8.6.1", "rich>=13.7.1",
        "python-dotenv>=1.0.1", "typing-extensions>=4.11.0", "requests",
        "anthropic>=0.47.0", "cohere>=5.3.4", "google-genai>=1.15.0")
pip("--no-deps", "-e", "vendor/agentdojo")
pip("--no-deps", "-e", ".")

# 4. Import check before spending GPU minutes on a broken environment.
out = run([virtual_env_python, "-c",
    "from agentdojo.task_suite.load_suites import get_suites;"
    "s=get_suites('v1.2.1');"
    "from harness.runner import AgentLoop, score_case;"
    "from harness.local_client import OpenAICompatClient;"
    "print('suites:', sorted(s))"])
print(out)

# 5. Snapshot these versions as a *candidate*. An import check is not enough to
#    call them good (the transformers 5.x break passed it), so the lock is only
#    written to Drive by cell 7, after a real model has loaded and called a tool.
CANDIDATE_LOCK = "/content/requirements-candidate.lock"
with open(CANDIDATE_LOCK, "w") as f:
    f.write(run([virtual_env_python, "-m", "pip", "freeze", "--exclude-editable"]))
print("transformers", run([virtual_env_python, "-c", "import transformers; print(transformers.__version__)"]).strip())

import json as _json
def baseline_done():
    """The Phase 1 result for this model + attack, if a complete one already exists."""
    f = f"analysis/out/phase1_en_baseline__{ATTACK}.json"
    if RERUN or not os.path.exists(f):
        return None
    d = _json.load(open(f, encoding="utf-8"))
    ok = (d.get("cell", {}).get("model") == SERVED and len(d.get("attack", [])) == 150
          and sum(r.get("errored", False) for r in d["attack"]) <= 15)
    return f if ok else None


## 6 · Serve the model

`--enable-auto-tool-choice` and `--tool-call-parser hermes` are both mandatory. Without
them vLLM hands tool calls back as prose, every case scores as a failed attack, and the
whole grid reads as *perfectly secure* — the exact failure this project exists to call out.

First boot downloads ~10 GB for the 14B, so give it a few minutes. The log is at
`/content/vllm.log` if anything goes wrong.

In [ ]:
import subprocess, time, requests, os, sys

LOG = "/content/vllm.log"

def serve(model, quant):
    cmd = [virtual_env_python, "-m", "vllm.entrypoints.openai.api_server",
           "--model", model,
           "--revision", REVISION,
           "--dtype", "half",                       # T4 has no bfloat16
           "--max-model-len", str(MAX_LEN),
           "--gpu-memory-utilization", str(GPU_UTIL),
           "--enable-auto-tool-choice",
           "--tool-call-parser", "hermes",          # Qwen2.5 speaks the Hermes format
           "--disable-log-requests",
           "--port", "8000"]
    if quant:
        cmd += ["--quantization", quant]
    cmd += EXTRA_ARGS
    print(" ".join(cmd), "\n")
    return subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

def wait(proc, minutes=25):
    deadline = time.time() + minutes * 60
    while time.time() < deadline:
        if proc.poll() is not None:
            return False
        try:
            if requests.get("http://localhost:8000/v1/models", timeout=3).ok:
                return True
        except Exception:
            pass
        time.sleep(10)
    return False

def tail(n=40):
    print("".join(open(LOG).readlines()[-n:]) if os.path.exists(LOG) else "(no log)")

# A server left over from an earlier run of this cell would answer the health
# check below and hide whether the new one actually started.
subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"])
time.sleep(5)

SERVED = MODEL
server = serve(MODEL, QUANT)
if wait(server):
    print("server up:", SERVED)
else:
    print(f"--- {MODEL} did not come up; log tail ---")
    tail()
    try: server.kill(); server.wait(timeout=30)
    except Exception: pass
    print(f"\nfalling back to {FALLBACK_MODEL}")
    SERVED = FALLBACK_MODEL
    server = serve(FALLBACK_MODEL, FALLBACK_QUANT)
    if not wait(server):
        tail()
        raise RuntimeError("no server. Read the log above before changing anything else.")
    print("server up on the fallback model. Record this in RESULTS.md.")

print("\nSERVED =", SERVED)

## 7 · Verify tool calling

The one check that must not be skipped, including on sessions where it passed before.
It costs seconds.

In [ ]:
try:
    from openai import OpenAI
except ImportError:
    import subprocess
    subprocess.run(["python3", "-m", "pip", "install", "-q", "openai"], check=True)
    from openai import OpenAI
c = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

r = c.chat.completions.create(
    model=SERVED,
    messages=[{"role": "user", "content": "What is my current account balance?"}],
    tools=[{"type": "function", "function": {
        "name": "get_balance",
        "description": "Get the account balance",
        "parameters": {"type": "object", "properties": {}},
    }}],
    tool_choice="auto",
    temperature=0.0,
    max_tokens=128,
)
calls = r.choices[0].message.tool_calls
print("tool_calls:", calls)
print("text:", repr(r.choices[0].message.content))
if not calls:
    tail(60)
    raise RuntimeError(
        "No tool call. Nothing downstream is meaningful. Check --enable-auto-tool-choice "
        "and --tool-call-parser hermes in the log above before running anything else."
    )
print("\nOK — the model calls tools through the OpenAI wire format.")

# The environment has now served a real model end to end: lock its versions.
import os, shutil
if LOCK and not os.path.exists(LOCK) and os.path.exists(CANDIDATE_LOCK):
    shutil.copy(CANDIDATE_LOCK, LOCK)
    print("locked known-good package versions to", LOCK)

## 8 · Close Phase 0

One real AgentDojo case, one real model call, traced and scored. This is the exit
criterion that has been open since week 1.

In [ ]:
import subprocess, sys
r = stream([virtual_env_python, "-m", "scripts.phase0_smoke",
                    "--backend", "local", "--model", SERVED])
print("\nexit code:", r.returncode)
if r.returncode:
    print("Phase 0 criterion NOT met. Read the output above — do not proceed to the "
          "baseline on a loop that cannot complete one case.")

## 9 · Pre-flight: 10 cases

The full run is long. Ten cases tell you the error rate, the mean latency and whether
the cache is filling — all the things you want to know *before* committing hours.

An error rate above ~10% almost always means tool calling is misconfigured, not that the
model is weak. `phase1_baseline` exits non-zero in that case.

In [ ]:
import time
if baseline_done():
    print("skipped: the full baseline for", SERVED, "/", ATTACK, "already exists (set RERUN = True to redo)")
else:
    t0 = time.time()
    r = stream([virtual_env_python, "-m", "scripts.phase1_baseline",
                "--backend", "local", "--model", SERVED, "--attack", ATTACK,
                "--workers", str(WORKERS), "--limit", str(SMOKE_N)])
    mins = (time.time() - t0) / 60
    print(f"\nexit {r.returncode} in {mins:.1f} min for {SMOKE_N} cases x 2 passes")
    print(f"extrapolated for 150 cases: ~{mins * 150 / SMOKE_N:.0f} min "
          f"(banking-heavy sample: the real run is usually longer)")
    if r.returncode:
        raise RuntimeError("pre-flight gate failed — fix before the full run")


## 10 · The baseline — all 150 cases

Two passes: the attack pass (injections in place) gives `ASR(en, undefended)`, which is
the denominator for every efficacy number in the paper; the clean pass gives
`Utility(en, undefended)`.

If Colab drops the session mid-run, re-run the notebook — the cache makes everything
already computed free, so the resumed run only does what is left. **Run cell 12 to push
the cache to Drive as soon as this finishes**, and again if you have to stop early.

In [ ]:
import time, shutil
done = baseline_done()
if done:
    print("skipped: using existing", done, "(set RERUN = True to redo)")
else:
    argv = [virtual_env_python, "-m", "scripts.phase1_baseline",
            "--backend", "local", "--model", SERVED, "--attack", ATTACK, "--workers", str(WORKERS)]
    if FULL_N:
        argv += ["--limit", str(FULL_N)]
    t0 = time.time()
    r = stream(argv)
    print(f"\nexit {r.returncode} after {(time.time()-t0)/60:.1f} min")
    if r.returncode:
        print("Above 10% of runs errored. The cell is not trustworthy — read the reasons "
              "printed above. Record the failure in RESULTS.md either way.")
    done = f"analysis/out/phase1_en_baseline__{ATTACK}.json"
# The probe joins onto the canonical file, so make sure it is this model + attack.
if os.path.exists(done):
    shutil.copy(done, "analysis/out/phase1_en_baseline.json")
save_to_drive("after the baseline")


## 11 · The English baseline on file


In [ ]:
import json
from pathlib import Path

# A short summary of the English baseline on file. It used to print a ready-made
# RESULTS.md entry every session, stale after the first run and labelled "direct"
# whatever the attack was. RESULTS.md is now written on the laptop from cache.tgz.
d = json.loads(Path("analysis/out/phase1_en_baseline.json").read_text())
attack, clean = d["attack"], d["clean"]
scored = [r for r in attack if not r.get("errored")]
print(f"served model   {SERVED}")
print(f"attack         {ATTACK}")
print(f"ASR (raw)      {sum(r['attacker_task_success'] for r in scored)}/{len(scored)}")
print(f"utility clean  {sum(r['user_task_success'] for r in clean)}/{len(clean)}")


## 11b · Phase 2 head start: the capability probe, all five languages

The model is already loaded, so this costs ~30-40 minutes rather than a second session. It
asks the agent to do each attacker goal **as a plain user request** and records whether it
could. The 150 cases share only 35 distinct goals, so each language is 35 runs.

English also joins onto the baseline above and prints the first real `uASR`, plus
*probe disagreements*: attacks that worked on goals the probe called impossible. Any of
those means the probe is too strict — look at them before trusting `C(en)`.

`hi`, `hi_latn`, `bn` and `mixed` run on **unreviewed machine drafts** (`corpus/goals/`),
so their numbers are a pilot: good enough to see whether C(L) falls off-English at all,
not good enough for the paper. Their output files are suffixed `_draft`.

Then run cell 12 — it saves all of this to Drive too.


In [ ]:
import json, os

def current_goals(lang):
    """(domain, injection_task, goal) for the goals the probe would ask today."""
    if lang == "en":
        rows = [json.loads(l) for l in open("corpus/injections/en.jsonl", encoding="utf-8")]
        return {(r["domain"], r["source_case_id"].split("/")[-1], r["attacker_goal_nl"]) for r in rows}
    rows = [json.loads(l) for l in open(f"corpus/goals/{lang}.jsonl", encoding="utf-8")]
    return {(r["domain"], r["injection_task"], r["goal"]) for r in rows}

def probe_done(lang):
    for name in (f"phase2_probe_{lang}.json", f"phase2_probe_{lang}_draft.json"):
        f = f"analysis/out/{name}"
        if os.path.exists(f) and not RERUN:
            d = json.load(open(f, encoding="utf-8"))
            have = {(g["domain"], g["injection_task"], g["goal"]) for g in d["goals"]}
            errored = sum(g.get("errored", False) for g in d["goals"])
            if d.get("model") == SERVED and have == current_goals(lang) and errored == 0:
                return f
    return None

def probe(lang, *extra):
    done = probe_done(lang)
    if done and lang != "en":
        print(f"[{lang}] skipped: {done} already matches {SERVED} and the current goals")
        return
    # en always re-runs (from cache, seconds) so it re-joins onto this attack's baseline.
    r = stream([virtual_env_python, "-m", "scripts.phase2_probe",
                "--lang", lang, "--model", SERVED, "--workers", str(WORKERS), *extra])
    print(f"\n[{lang}] exit code:", r.returncode, "\n" + "-" * 70)

probe("en")
# PILOT: the four other languages use machine-drafted goals that have not been
# human-reviewed yet. Their C(L) is an early signal for the go/no-go, not a result.
for lang in ("hi", "hi_latn", "bn", "mixed"):
    probe(lang, "--allow-unverified")
save_to_drive("after the probe")


## 11b2 · C3 check: the Hindi probe again, in a Hindi environment

Everything above runs in AgentDojo's English world -- only the attacker's goal is Hindi.
`--localized-env` also translates the environment's prose (emails, documents, reviews,
the banking files; names, amounts and ids stay as they are, because the checkers compare
them -- see `corpus/environments.py`). Same 35 goals, ~10 minutes.

The question: does C(hi) move when the inbox is Hindi too? If it does, every
English-environment multilingual number in the literature is measuring a world no Hindi
user lives in. Output: `phase2_probe_hi_locenv.json`, saved to Drive with the rest.


In [ ]:
import os, json
f = "analysis/out/phase2_probe_hi_locenv.json"
done = os.path.exists(f) and not RERUN and json.load(open(f, encoding="utf-8")).get("model") == SERVED
if done:
    print("skipped:", f, "already exists for", SERVED)
else:
    r = stream([virtual_env_python, "-m", "scripts.phase2_probe", "--lang", "hi", "--localized-env",
                "--model", SERVED, "--workers", str(WORKERS)])
    print("\nexit code:", r.returncode)
save_to_drive("hi probe, localized environment")


## 11c · The grid, in priority order

One step = one grid cell. Finished cells are skipped, so **re-run this cell every session**
until it prints the two summary tables with every row filled. **It is not meant to finish in
one session**: the whole plan is ~10 GPU hours and a free session gives ~4. When Colab cuts
the session off, that is fine -- progress is saved to Drive **every 15 minutes** (and after
every step), so the next session resumes where this one stopped, from the response cache.

Order, decided 2026-10-07 from the first grid results (see `RESULTS.md`):

1. **Hindi undefended, all 150 cases** -- raw ASR vs uASR in Hindi, the paper's core contrast (~65 min).
2. English cells, with the architectural ones (Class C) before the second prompting one.
3. **Hindi defended cells on the scorable set only** (`--scorable-only`): C(hi) ~ 0.27, so
   only ~40 of 150 cases can count toward uASR or efficacy, and the rest are skipped (~15-20 min each).
4. `hierarchy` is dropped (2026-10-07): sandwich and spotlight already represent prompting defenses.

Hindi steps run only once the Hindi attack template is human-verified (the grid refuses
machine drafts). Until then they are skipped with a message and the English steps still run.


In [ ]:
import os, time, subprocess, threading

# Autosave: a step can run 1.5 h, and a session cut off mid-step used to lose all of it
# (spotlight, 2026-10-07). Every 15 min the response cache goes to Drive, so a killed step
# resumes from cache next session. One lock, so the autosave and the end-of-step save
# never write /content/cache.tgz at the same time.
if not getattr(save_to_drive, "_locked", False):
    _save_lock, _save_unlocked = threading.Lock(), save_to_drive
    def save_to_drive(note=""):
        with _save_lock:
            _save_unlocked(note)
    save_to_drive._locked = True
_autosave_stop = threading.Event()
def _autosave():
    while not _autosave_stop.wait(15 * 60):
        save_to_drive("autosave")
threading.Thread(target=_autosave, daemon=True).start()
pip("sentencepiece", "protobuf")   # DeBERTa-v3 tokenizer, for the protectai steps

PLAN = [
    ("hi", "undefended"),          # all 150 cases: raw ASR + the existence proof need them
    ("en", "spotlight"),
    ("hi", "sandwich"),
    ("en", "camel_lite"),
    ("hi", "camel_lite"),
    ("en", "allowlist"),
    ("en", "protectai_deberta"),   # Class A in the loop, default threshold 0.5
    ("hi", "protectai_deberta"),
    ("en", "honeytool"),
    # ("en", "hierarchy"),         # dropped 2026-10-07 (de-scoping ladder): sandwich + spotlight
                                   # already cover Class B; re-enable only if time is left over
]

r = subprocess.run([virtual_env_python, "-c",
                    "from corpus.attacks import TEMPLATE_STATUS as T; print(bool(T['hi']['verified_by']))"],
                   capture_output=True, text=True)
HI_READY = r.stdout.strip() == "True"
if not HI_READY:
    print("Hindi steps SKIPPED: the Hindi attack template is not human-verified yet "
          "(corpus/review/templates.txt). English steps still run.\n")

for lang, d in PLAN:
    if lang == "hi" and not HI_READY:
        continue
    if os.path.exists(f"analysis/out/grid/{lang}__{d}__{ATTACK}.json") and not RERUN:
        continue
    argv = [virtual_env_python, "-m", "scripts.phase5_grid", "--langs", lang, "--defenses", d,
            "--model", SERVED, "--attack", ATTACK, "--workers", str(WORKERS),
            "--classifier-device", "cpu"]
    if lang != "en":
        argv.append("--scorable-only")
    t0 = time.time()
    r = stream(argv)
    print(f"[{lang} {d}] exit {r.returncode} after {(time.time() - t0) / 60:.1f} min")
    save_to_drive(f"grid {lang} {d}")

for lang in ("en", "hi"):
    ds = ["undefended"] + [d for l, d in PLAN if l == lang and d != "undefended"]
    if lang == "en":
        ds.insert(1, "sandwich")
    stream([virtual_env_python, "-m", "scripts.phase5_grid", "--summary-only",
            "--langs", lang, "--defenses", *ds, "--attack", ATTACK])

_autosave_stop.set()


## 12 · Save everything off the machine

The baseline and probe cells already save to Drive when they finish; this is the final, belt-and-braces save. Run it before you stop, even if a cell above failed.


In [ ]:
save_to_drive("final")
if not DRIVE_OK:
    from google.colab import files
    files.download("/content/cache.tgz")


## After this notebook

Download `MyDrive/agentfw/cache.tgz` into `Desktop/LLM/agentfw/` on the laptop. Every table,
figure and `RESULTS.md` entry is regenerated there from the files inside it, so nothing from
this notebook needs to be copied or pasted by hand.
